# **NumPy Lab Guide: The Basics with the WFP Nigeria Food Price Data**

This lab follows the order of the [W3Schools NumPy tutorial](https://www.w3schools.com/python/numpy/), from *Creating Arrays* to *Array Filter*, then the most useful *ufuncs*. Every example uses real food prices from Nigerian markets in `wfp_food_prices_nga.csv`.

**How each section works:** a short explanation, then one or more **problems**. Type each code cell yourself, run it, and compare your output with the one shown.

| Section | Topic |
|---|---|
| 1 | Creating Arrays |
| 2 | Array Indexing |
| 3 | Array Slicing |
| 4 | Data Types |
| 5 | Copy vs View |
| 6 | Array Shape |
| 7 | Array Reshape |
| 8 | Array Iterating |
| 9 | Array Join |
| 10 | Array Split |
| 11 | Array Search |
| 12 | Array Sort |
| 13 | Array Filter |
| 14 | ufuncs: arithmetic, rounding, sums, differences, sets |
| 15 | Exercises |

## **Setup: load the data**

We use pandas **only to read the file**. Everything after that is NumPy: we take a column out of the DataFrame with `np.array(df["column"])` and work on the array.

The file is not stored in date order, so we sort it once by date. Many problems below depend on prices being in time order.

In [1]:
import numpy as np
import pandas as pd

print(f"NumPy version: {np.__version__}")

df = pd.read_csv("wfp_food_prices_nga.csv")
df = df.sort_values("date", kind="stable").reset_index(drop=True)   # put rows in date order

print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

NumPy version: 2.5.3
Rows: 90193, Columns: 16


In [2]:
df.head()

,date,admin1,admin2,market,market_id,latitude,longitude,category,commodity,commodity_id,unit,priceflag,pricetype,currency,price,usdprice
0,2002-01-15,Katsina,Jibia,Jibia (CBM),1038,13.08,7.24,cereals and tubers,Maize,51,KG,actual,Wholesale,NGN,175.92,1.54
1,2002-01-15,Katsina,Jibia,Jibia (CBM),1038,13.08,7.24,cereals and tubers,Millet,73,KG,actual,Wholesale,NGN,150.18,1.31
2,2002-01-15,Katsina,Jibia,Jibia (CBM),1038,13.08,7.24,cereals and tubers,Rice (imported),64,KG,actual,Wholesale,NGN,358.70,3.14
3,2002-01-15,Katsina,Jibia,Jibia (CBM),1038,13.08,7.24,cereals and tubers,Sorghum,65,KG,actual,Wholesale,NGN,155.61,1.36
4,2002-01-15,Katsina,Jibia,Jibia (CBM),1038,13.08,7.24,pulses and nuts,Beans (niebe),120,KG,actual,Wholesale,NGN,196.87,1.72


### **Three slices of the data we will use often**

1. **`monday`**: the retail price of imported rice (per 2.8 kg measure) at Monday Market, Maiduguri, month by month.
2. **`rice25`**: every 2025 retail price of imported rice, from all markets.
3. **`all_prices`**: every price in the file.

In [3]:
m = df[(df["commodity"] == "Rice (imported)") & (df["pricetype"] == "Retail") & (df["market"] == "Monday")]
monday = np.array(m["price"])                 # np.array() makes our own copy
monday_dates = np.array(m["date"])

r = df[(df["commodity"] == "Rice (imported)") & (df["pricetype"] == "Retail") & (df["date"].str.startswith("2025"))]
rice25 = np.array(r["price"])
rice25_markets = np.array(r["market"])

all_prices = np.array(df["price"])

print("Monday Market prices:", monday.size, "| from", monday_dates[0], "to", monday_dates[-1])
print("2025 rice prices:", rice25.size)
print("All prices:", all_prices.size)

Monday Market prices: 105 | from 2016-06-15 to 2026-08-15
2025 rice prices: 289
All prices: 90193


# **1. Creating Arrays**

NumPy's array object is called **`ndarray`**. You create one with `np.array()`, passing a **list**, a **tuple** or any array-like object, such as a pandas column.

**Dimensions** are levels of nesting:

| Array | Example | `ndim` |
|---|---|---|
| 0-D (a scalar) | one price | 0 |
| 1-D | a list of prices | 1 |
| 2-D | a table: markets × months | 2 |
| 3-D | a stack of tables: commodities × markets × months | 3 |

## **Problem 1: Create arrays from a list and a tuple**
The first three rows of the file are January 2002 wholesale prices per kg in Jibia: maize N175.92, millet N150.18 and imported rice N358.70.

In [4]:
from_list = np.array([175.92, 150.18, 358.70])
from_tuple = np.array((175.92, 150.18, 358.70))

print(from_list)
print(from_tuple)
print(type(from_list))

[175.92 150.18 358.7 ]
[175.92 150.18 358.7 ]
<class 'numpy.ndarray'>


## **Problem 2: 0-D, 1-D, 2-D and 3-D arrays from the dataset**
Build one array of each dimension from Monday, Custom and Budum markets in Maiduguri (imported and local rice, January to April 2025).

In [5]:
markets = ["Monday", "Custom", "Budum"]

def first4(commodity, market):
    sel = df[(df["commodity"] == commodity) & (df["pricetype"] == "Retail")
             & (df["market"] == market) & (df["date"].str.startswith("2025"))]
    return sel["price"].to_numpy()[:4]          # January to April 2025

price_0d = np.array(monday[0])                                     # one price
price_1d = first4("Rice (imported)", "Monday")                     # 4 months
price_2d = np.array([first4("Rice (imported)", mk) for mk in markets])   # 3 markets x 4 months
price_3d = np.array([[first4(c, mk) for mk in markets]
                     for c in ["Rice (imported)", "Rice (local)"]])       # 2 x 3 x 4

for name, a in [("0-D", price_0d), ("1-D", price_1d), ("2-D", price_2d), ("3-D", price_3d)]:
    print(f"{name}: ndim = {a.ndim}, shape = {a.shape}")
print(price_2d)

0-D: ndim = 0, shape = ()
1-D: ndim = 1, shape = (4,)
2-D: ndim = 2, shape = (3, 4)
3-D: ndim = 3, shape = (2, 3, 4)
[[8963.16 9055.36 8750.   8100.  ]
 [7000.   6883.93 6653.23 7388.89]
 [6500.   6285.71 5717.74 6000.  ]]


## **Problem 3: Turn a DataFrame column into an array, and force extra dimensions**
There are two ways to get a column out as an array. `.to_numpy()` may give a **read-only** view of pandas' own data (pandas 3 protects it), while `np.array()` always makes a new array you can change.

In [6]:
view = df["price"].to_numpy()
own = np.array(df["price"])
print(type(own), own.ndim, own.shape)
print("to_numpy() writeable:", view.flags.writeable)
print("np.array() writeable:", own.flags.writeable)

five_d = np.array(price_1d, ndmin=5)     # ndmin adds empty dimensions in front
print(five_d)
print("ndim:", five_d.ndim)

<class 'numpy.ndarray'> 1 (90193,)
to_numpy() writeable: False
np.array() writeable: True
[[[[[8963.16 9055.36 8750.   8100.  ]]]]]
ndim: 5


# **2. Array Indexing**

Indexing gets **one element**. Positions start at **0**. Negative positions count from the end: `-1` is the last.

In 2-D, write `arr[row, column]`. In 3-D, write `arr[table, row, column]`.

## **Problem 4: First, last and selected prices**

In [7]:
print("First Monday price:", monday_dates[0], monday[0])
print("Third Monday price:", monday_dates[2], monday[2])
print("Last Monday price: ", monday_dates[-1], monday[-1])
print("Sum of first two:  ", monday[0] + monday[1])

First Monday price: 2016-06-15 966.0
Third Monday price: 2016-08-15 1100.0
Last Monday price:  2026-08-15 6750.0
Sum of first two:   1966.0


## **Problem 5: Index the 2-D and 3-D price tables**

In [8]:
print("Custom Market, February 2025:", price_2d[1, 1])
print("Budum Market, last month:     ", price_2d[-1, -1])
print("Local rice, Monday, March:    ", price_3d[1, 0, 2])

Custom Market, February 2025: 6883.93
Budum Market, last month:      6000.0
Local rice, Monday, March:     4000.0


# **3. Array Slicing**

Slicing takes **a range**: `arr[start:end:step]`. The `start` position is included and the `end` is not. Leave out `start` for "from the beginning", `end` for "to the end", and `step` for 1.

## **Problem 6: Slices of the Monday Market history**

In [9]:
print("First 6 months:    ", monday[:6])
print("Positions 4 to 7:  ", monday[4:8])
print("Last 3 records:    ", monday[-3:])
print("Every 12th record: ", monday[:72:12])        # June 2016, June 2017, ...
print("Matching dates:    ", monday_dates[:72:12])

First 6 months:     [ 966. 1000. 1100. 1088. 1200. 1200.]
Positions 4 to 7:   [1200. 1200. 1210. 1220.]
Last 3 records:     [7533.33 7578.95 6750.  ]
Every 12th record:  [ 966. 1200. 1112. 1200. 1700. 1850.]
Matching dates:     ['2016-06-15' '2017-06-15' '2018-06-15' '2019-06-15' '2020-06-15'
 '2021-06-15']


## **Problem 7: Slicing a 2-D table**

In [10]:
print("Monday and Custom, Jan to Mar:")
print(price_2d[0:2, 0:3])
print("Every market, January:", price_2d[:, 0])
print("Budum, Feb to Apr:    ", price_2d[2, 1:])

Monday and Custom, Jan to Mar:
[[8963.16 9055.36 8750.  ]
 [7000.   6883.93 6653.23]]
Every market, January: [8963.16 7000.   6500.  ]
Budum, Feb to Apr:     [6285.71 5717.74 6000.  ]


# **4. Data Types**

Every array has **one data type**, shown by `.dtype`. The common codes are:

| Code | Type | Code | Type |
|---|---|---|---|
| `i` | integer | `b` | boolean |
| `u` | unsigned integer | `M` | datetime |
| `f` | float | `O` | object |
| `U` | unicode string | `S` | byte string |

Use `.astype()` to convert an array to another type. It returns a **new** array.

## **Problem 8: What type is each column?**

In [11]:
for col in ["price", "market_id", "latitude", "market"]:
    arr = df[col].to_numpy()
    print(f"{col:10} -> {arr.dtype}")

names = np.array(["Monday", "Custom", "Budum"])
print("A string array:", names.dtype)

price      -> float64
market_id  -> int64
latitude   -> float64
market     -> object
A string array: <U6


## **Problem 9: Converting types**
Prices typed as text cannot be used in arithmetic until they are converted. Dates as text can be converted to real dates.

In [12]:
as_text = np.array(["175.92", "150.18", "358.70"])
as_float = as_text.astype(float)
print(as_text.dtype, "->", as_float.dtype, "| total:", as_float.sum())

as_int = as_float.astype("i")              # decimals are cut off, not rounded
print("As integers:", as_int)

dates = monday_dates[:3].astype("datetime64[D]")
print("Dates:", dates, dates.dtype)
print("Days between first two records:", dates[1] - dates[0])

<U6 -> float64 | total: 684.8
As integers: [175 150 358]
Dates: ['2016-06-15' '2016-07-15' '2016-08-15'] datetime64[D]
Days between first two records: 30 days


## **Problem 10: When conversion fails**
A lecturer or data clerk may type `abs` instead of a number. `astype(float)` then raises an error.

In [13]:
raw = np.array(["5000", "abs", "6200"])
try:
    raw.astype(float)
except ValueError as e:
    print("ValueError:", e)

ValueError: could not convert string to float: np.str_('abs')


# **5. Copy vs View**

* A **copy** is a new array. Changing it does not affect the original.
* A **view** shares the original's data. Changing one changes the other. **Slices are views.**

Check with `.base`: a copy has `base = None`; a view's `base` is the original array.

## **Problem 11: Change a copy, then a view**

In [14]:
first4_copy = monday[:4].copy()
first4_view = monday[:4]

first4_copy[0] = 0
print("After changing the copy, monday[0] =", monday[0])

first4_view[0] = 0
print("After changing the view, monday[0] =", monday[0])

monday[0] = 966.0                          # put the real value back
print("Restored monday[0] =", monday[0])
print("copy.base is None:", first4_copy.base is None, "| view.base is None:", first4_view.base is None)

After changing the copy, monday[0] = 966.0
After changing the view, monday[0] = 0.0
Restored monday[0] = 966.0
copy.base is None: True | view.base is None: False


# **6. Array Shape**

`.shape` returns a tuple with the number of elements in each dimension. The number of values in the tuple equals `ndim`.

## **Problem 12: Shapes of our arrays**

In [15]:
print("All prices:", all_prices.shape)
print("Monday series:", monday.shape)
print("2-D table:", price_2d.shape)
print("3-D table:", price_3d.shape)
print("ndmin=5 array:", five_d.shape)

All prices: (90193,)
Monday series: (105,)
2-D table: (3, 4)
3-D table: (2, 3, 4)
ndmin=5 array: (1, 1, 1, 1, 4)


# **7. Array Reshape**

`.reshape()` changes the shape without changing the data. The number of elements must stay the same: 72 values can become 6 × 12, never 5 × 12. Use `-1` to let NumPy work out one dimension, and `.reshape(-1)` to flatten back to 1-D.

## **Problem 13: Six years of monthly prices as a table**
Monday Market has an unbroken run of 72 monthly prices from June 2016 to May 2022. Reshape it to one row per year (June to May).

In [16]:
print("Position 0:", monday_dates[0], "| position 71:", monday_dates[71])

years = monday[:72].reshape(6, 12)
print(years)
print("Shape:", years.shape)

Position 0: 2016-06-15 | position 71: 2022-05-15
[[ 966. 1000. 1100. 1088. 1200. 1200. 1210. 1220. 1100. 1200. 1200. 1206.]
 [1200. 1200. 1966. 1150. 1150. 1100. 1100. 1000. 1150. 1200. 1200. 1100.]
 [1112. 1100. 1200. 1200. 1200. 1200. 1160. 1200. 1200. 1200. 1200. 1200.]
 [1200. 1166. 1150. 1160. 1216. 1258. 1300. 1322. 1600. 1600. 1734. 1760.]
 [1700. 1712. 1700. 1716. 1216. 1726. 1734. 1750. 1764. 1800. 1950. 1938.]
 [1850. 1800. 1850. 1850. 1900. 1950. 1850. 1850. 1900. 1900. 2000. 2000.]]
Shape: (6, 12)


## **Problem 14: Using -1, flattening, and an impossible reshape**

In [17]:
quarters = monday[:12].reshape(-1, 3)        # NumPy works out 4 rows
print("Quarters:\n", quarters)

flat = years.reshape(-1)
print("Flattened shape:", flat.shape)
print("Is reshape a view?", years.base is not None)

try:
    monday[:72].reshape(5, 12)
except ValueError as e:
    print("ValueError:", e)

Quarters:
 [[ 966. 1000. 1100.]
 [1088. 1200. 1200.]
 [1210. 1220. 1100.]
 [1200. 1200. 1206.]]
Flattened shape: (72,)
Is reshape a view? True
ValueError: cannot reshape array of size 72 into shape (5,12)


# **8. Array Iterating**

A `for` loop over a 1-D array visits each element; over a 2-D array, each **row**. `np.nditer()` visits every element in any number of dimensions, and `np.ndenumerate()` also gives each element's position.

## **Problem 15: Loop over markets and months**

In [18]:
for mk, row in zip(markets, price_2d):
    print(f"{mk:7} average Jan to Apr: N{row.mean():,.2f}")

print("Every price in the 2-D table:")
for x in np.nditer(price_2d):
    print(x, end="  ")
print()

months = ["Jan", "Feb", "Mar", "Apr"]
for (i, j), x in np.ndenumerate(price_2d):
    if x > 8000:
        print(f"Above N8,000: {markets[i]} in {months[j]}: N{x:,.2f}")

Monday  average Jan to Apr: N8,717.13
Custom  average Jan to Apr: N6,981.51
Budum   average Jan to Apr: N6,125.86
Every price in the 2-D table:
8963.16  9055.36  8750.0  8100.0  7000.0  6883.93  6653.23  7388.89  6500.0  6285.71  5717.74  6000.0  
Above N8,000: Monday in Jan: N8,963.16
Above N8,000: Monday in Feb: N9,055.36
Above N8,000: Monday in Mar: N8,750.00
Above N8,000: Monday in Apr: N8,100.00


# **9. Array Join**

* `np.concatenate((a, b))` joins along an existing axis (end to end for 1-D).
* `np.stack((a, b))` joins along a **new** axis.
* Helpers: `np.hstack` (side by side), `np.vstack` (on top of each other) and `np.dstack` (along depth).

## **Problem 16: Join Monday and Custom markets**

In [19]:
monday4 = price_2d[0]
custom4 = price_2d[1]

print("concatenate:", np.concatenate((monday4, custom4)))
print("stack (2 x 4):\n", np.stack((monday4, custom4)))
print("stack axis=1 (4 x 2):\n", np.stack((monday4, custom4), axis=1))
print("hstack:", np.hstack((monday4, custom4)).shape,
      "| vstack:", np.vstack((monday4, custom4)).shape,
      "| dstack:", np.dstack((monday4, custom4)).shape)

concatenate: [8963.16 9055.36 8750.   8100.   7000.   6883.93 6653.23 7388.89]
stack (2 x 4):
 [[8963.16 9055.36 8750.   8100.  ]
 [7000.   6883.93 6653.23 7388.89]]
stack axis=1 (4 x 2):
 [[8963.16 7000.  ]
 [9055.36 6883.93]
 [8750.   6653.23]
 [8100.   7388.89]]
hstack: (8,) | vstack: (2, 4) | dstack: (1, 4, 2)


# **10. Array Split**

`np.array_split(arr, n)` cuts an array into n parts. Unlike `np.split`, it works even when the array does not divide evenly. `np.hsplit` and `np.vsplit` split tables by columns or rows.

## **Problem 17: Split a year into seasons, and the 6-year table into halves**

In [20]:
one_year = years[0]                          # June 2016 to May 2017
seasons = np.array_split(one_year, 4)
for name, part in zip(["Jun-Aug", "Sep-Nov", "Dec-Feb", "Mar-May"], seasons):
    print(f"{name}: {part}")

uneven = np.array_split(monday[:10], 3)       # 10 values into 3 parts
print("Uneven split sizes:", [len(p) for p in uneven])

first_half, second_half = np.hsplit(years, 2)   # columns Jun-Nov and Dec-May
print("hsplit shapes:", first_half.shape, second_half.shape)

Jun-Aug: [ 966. 1000. 1100.]
Sep-Nov: [1088. 1200. 1200.]
Dec-Feb: [1210. 1220. 1100.]
Mar-May: [1200. 1200. 1206.]
Uneven split sizes: [4, 3, 3]
hsplit shapes: (6, 6) (6, 6)


# **11. Array Search**

* `np.where(condition)` returns the **positions** where the condition is True.
* `np.searchsorted(sorted_arr, value)` returns where a value would be inserted to keep a **sorted** array in order.

## **Problem 18: When was Monday Market rice above N7,000?**

In [21]:
pos = np.where(monday > 7000)[0]
print("Positions:", pos)
print("Dates:", monday_dates[pos])
print("Number of months:", pos.size)

Positions: [ 82  84  85  86  87  88  89  90  91  92  93  95 100 101 102 103]
Dates: ['2024-01-15' '2024-10-15' '2024-11-15' '2024-12-15' '2025-01-15'
 '2025-02-15' '2025-03-15' '2025-04-15' '2025-05-15' '2025-08-15'
 '2025-09-15' '2025-11-15' '2026-04-15' '2026-05-15' '2026-06-15'
 '2026-07-15']
Number of months: 16


## **Problem 19: Where would a new price fit?**

In [22]:
sorted_rice = np.sort(rice25)
i = np.searchsorted(sorted_rice, 5000)
print(f"N5,000 would go at position {i} of {sorted_rice.size}")
print(f"So {i} of the 2025 prices are below N5,000 ({i / sorted_rice.size:.0%})")

N5,000 would go at position 130 of 289
So 130 of the 2025 prices are below N5,000 (45%)


# **12. Array Sort**

`np.sort()` returns a sorted **copy**: numbers in ascending order, strings alphabetically, booleans with False first. On a 2-D array it sorts each row.

## **Problem 20: Sort prices, market names and a table**

In [23]:
print("5 lowest 2025 prices: ", np.sort(rice25)[:5])
print("5 highest 2025 prices:", np.sort(rice25)[::-1][:5])

unique_markets = np.unique(rice25_markets)
print("First 5 markets, A to Z:", np.sort(unique_markets)[:5])

print("Each market's prices sorted:\n", np.sort(price_2d))

5 lowest 2025 prices:  [2338.71 2388.46 2458.33 2464.52 2470.97]
5 highest 2025 prices: [9091.3  9055.36 8963.16 8926.32 8800.  ]
First 5 markets, A to Z: ['Abba Gamaram' 'Bade (Gashua)' 'Baga Road' 'Bama' 'Banki']
Each market's prices sorted:
 [[8100.   8750.   8963.16 9055.36]
 [6653.23 6883.93 7000.   7388.89]
 [5717.74 6000.   6285.71 6500.  ]]


# **13. Array Filter**

Filtering picks some elements and leaves others out, using a **boolean index list**: an array of True and False the same length as the data. Usually you build it straight from a condition: `arr[arr > 7000]`.

Combine conditions with `&` (and), `|` (or) and `~` (not), with each condition in brackets.

## **Problem 21: A filter built by hand, then from a condition**

In [24]:
x = price_2d[0]                             # Monday Market, Jan to Apr
by_hand = [True, True, False, False]
print("By hand:", x[by_hand])

high = x > 8800
print("Filter:", high)
print("Prices above N8,800:", x[high])

By hand: [8963.16 9055.36]
Filter: [ True  True False False]
Prices above N8,800: [8963.16 9055.36]


## **Problem 22: Filter the whole dataset**
Using arrays only, find the 2025 retail prices of imported rice in Yobe and compare them with Borno.

In [25]:
commodity = df["commodity"].to_numpy()
state = df["admin1"].to_numpy()
year = df["date"].str[:4].astype(int).to_numpy()
ptype = df["pricetype"].to_numpy()

base = (commodity == "Rice (imported)") & (ptype == "Retail") & (year == 2025)
for st in ["Borno", "Yobe"]:
    sel = all_prices[base & (state == st)]
    print(f"{st:6} records: {sel.size:3}  median: N{np.median(sel):,.2f}")

Borno  records: 171  median: N5,760.42
Yobe   records: 108  median: N4,356.74


# **14. ufuncs: Fast Element-by-Element Operations**

**ufuncs** (universal functions) work on every element at once, in fast compiled code, instead of a Python `for` loop. This is called **vectorization**.

| Group | Functions |
|---|---|
| Arithmetic | `np.add`, `np.subtract`, `np.multiply`, `np.divide`, `np.power`, `np.absolute` |
| Rounding | `np.around`, `np.floor`, `np.ceil`, `np.trunc` |
| Summations | `np.sum`, `np.cumsum` |
| Differences | `np.diff` |
| Sets | `np.unique`, `np.union1d`, `np.intersect1d`, `np.setdiff1d` |

## **Problem 23: Arithmetic: price per kg and in US dollars**
Imported rice is sold per 2.8 kg measure. Convert Monday Market's Jan to Apr 2025 prices to Naira per kg, then to US dollars at N1,550 per dollar.

In [26]:
per_kg = np.divide(monday4, 2.8)
print("Per kg:", per_kg)
print("Per kg in USD:", np.divide(per_kg, 1550))
print("Monday minus Custom:", np.subtract(monday4, custom4))

Per kg: [3201.12857143 3234.05714286 3125.         2892.85714286]
Per kg in USD: [2.06524424 2.08648848 2.01612903 1.86635945]
Monday minus Custom: [1963.16 2171.43 2096.77  711.11]


## **Problem 24: Rounding**

In [27]:
print("around (2 dp):", np.around(per_kg, 2))
print("floor:        ", np.floor(per_kg))
print("ceil:         ", np.ceil(per_kg))
print("trunc:        ", np.trunc(per_kg))

around (2 dp): [3201.13 3234.06 3125.   2892.86]
floor:         [3201. 3234. 3125. 2892.]
ceil:          [3202. 3235. 3125. 2893.]
trunc:         [3201. 3234. 3125. 2892.]


## **Problem 25: Summations: total and running total**
A household buys one measure of rice every month from June 2016. How much has it spent after each month of the first year?

In [28]:
first_year = monday[:12]
print("Total for the year:", np.sum(first_year))
print("Running total:", np.cumsum(first_year))
print("Yearly totals (6 years):", np.sum(years, axis=1))

Total for the year: 13690.0
Running total: [  966.  1966.  3066.  4154.  5354.  6554.  7764.  8984. 10084. 11284.
 12484. 13690.]
Yearly totals (6 years): [13690. 14516. 14172. 16466. 20706. 22700.]


## **Problem 26: Differences: month-to-month change**

In [29]:
change = np.diff(monday[:12])
print("Change in Naira:", change)

pct = np.around(np.diff(monday[:12]) / monday[:11] * 100, 1)
print("Change in %:    ", pct)
print("Biggest jump:", pct.max(), "% | biggest fall:", pct.min(), "%")

Change in Naira: [  34.  100.  -12.  112.    0.   10.   10. -120.  100.    0.    6.]
Change in %:     [ 3.5 10.  -1.1 10.3  0.   0.8  0.8 -9.8  9.1  0.   0.5]
Biggest jump: 10.3 % | biggest fall: -9.8 %


## **Problem 27: Set operations: which markets sell both kinds of rice?**

In [30]:
local = df[(df["commodity"] == "Rice (local)") & (df["pricetype"] == "Retail")
           & (df["date"].str.startswith("2025"))]["market"].to_numpy()

imported_mk = np.unique(rice25_markets)
local_mk = np.unique(local)

both = np.intersect1d(imported_mk, local_mk)
print("Markets with imported rice:", imported_mk.size)
print("Markets with local rice:   ", local_mk.size)
print("Markets with both:         ", both.size)
print("Imported only:", np.setdiff1d(imported_mk, local_mk))
print("Local only:   ", np.setdiff1d(local_mk, imported_mk))
print("All markets:  ", np.union1d(imported_mk, local_mk).size)

Markets with imported rice: 42
Markets with local rice:    44
Markets with both:          42
Imported only: []
Local only:    ['Biriri' 'Gulani (Tettaba)']
All markets:   44


# **15. Exercises**

Use NumPy for every answer, after loading the data with pandas.

1. **Creating arrays.** Make a 1-D array of the Custom Market imported rice prices, and print its `ndim`, `shape` and `dtype`.
2. **Indexing and slicing.** Print the December prices of 2016 to 2021 from the Monday series. (Hint: the series starts in June 2016, so the first December is at position 6.)
3. **Data types.** Convert the `date` column of the 2025 rice records to `datetime64[D]`. How many days are there between the first and last record?
4. **Copy vs view.** Take `years[2]` (one row of the 6-year table) and change its first value. Did `monday` change? Explain why.
5. **Reshape.** Reshape the 72 Monday prices into 24 quarters of 3 months. Which quarter had the highest total?
6. **Join and split.** Stack the Jan to Apr 2025 prices of Monday, Custom and Budum with `np.vstack`, then split the result into its three rows with `np.vsplit`.
7. **Search.** Find the positions and dates where the Monday price jumped by more than 40% in a single month. (Hint: `np.diff` and `np.where`.)
8. **Sort.** Find the 10 cheapest 2025 rice prices and the markets where they were recorded. (Hint: `np.argsort`.)
9. **Filter.** What share of all 2025 rice prices were above N6,000?
10. **ufuncs.** Repeat Problem 27 for millet and sorghum. Which markets sell one but not the other?